In [ ]:
# If running from a clean development environment, install the local package once:
# %pip install -e ..


# Polarized Trees: Benchmark, Model Selection, and Inference

This notebook evaluates Polarized Trees on the fixed synthetic corpora generated in the dataset-generation notebook.

The benchmark:

1. samples 800 configurations from the full 3,240-point hyperparameter grid;
2. evaluates each configuration on the three synthetic corpora;
3. compares configurations using mean Jaccard across the corpora;
4. selects the best-performing configuration;
5. examines the behavior of the selected and top-performing configurations;
6. applies the selected configuration to a separate unseen synthetic corpus in inference mode.

The synthetic benchmark configuration is defined once in `benchmark_config.py` and is shared with the dataset-generation notebook. The benchmark itself does not regenerate or redefine the synthetic corpora.

## 1. Imports and experiment settings

In [3]:
import itertools
import json
import os
import random
import sys
import time
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from polartox.polarized_trees import PolarizedTreesPipeline


# ------------------------------------------------------------
# Locate project root
# ------------------------------------------------------------

CWD = Path.cwd().resolve()

if (CWD / "benchmark_config.py").exists():
    PROJECT_ROOT = CWD
elif (CWD.parent / "benchmark_config.py").exists():
    PROJECT_ROOT = CWD.parent
else:
    raise FileNotFoundError(
        "Could not locate project root. "
        "Expected benchmark_config.py in the current "
        "directory or its parent."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from benchmark_config import (
    DIMS_DICT,
    DIMS,
    SCALE,
    CORPUS_CONFIGS,
    INFERENCE_CONFIG,
    BENCHMARK_CORPORA,
    INFERENCE_CORPUS,
)


# ------------------------------------------------------------
# Data and output directories
# ------------------------------------------------------------

DATA_DIR = PROJECT_ROOT / "benchmark_data"
RESULTS_DIR = PROJECT_ROOT / "benchmark_results"

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Benchmark data directory not found: {DATA_DIR}. "
        "Run datasetdemo.ipynb first."
    )

RESULTS_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Benchmark settings
# ------------------------------------------------------------

N_SAMPLES = 800
RANDOM_SEED = 0

print("Project root:", PROJECT_ROOT)
print("Dimensions:", DIMS)
print("Benchmark corpora:", BENCHMARK_CORPORA)
print("Inference corpus:", INFERENCE_CORPUS)
print("Data directory:", DATA_DIR)
print("Results directory:", RESULTS_DIR)


## 2. Load the fixed corpora

The dataset-generation notebook creates the synthetic corpora once using the shared configuration and stores the resulting datasets and ground truth in `benchmark_data/`.

Here we only load those fixed datasets. The same corpora are therefore used unchanged across every hyperparameter configuration, so differences in recovery performance are attributable to the Polarized Trees configuration rather than to resampling the synthetic data.

In [15]:
def load_corpus(name):
    dataset = pd.read_csv(
        DATA_DIR / f"{name}_dataset.csv"
    )

    with open(
        DATA_DIR / f"{name}_ground_truth.json"
    ) as f:
        ground_truth_raw = json.load(f)

    ground_truth = {
        int(text_id): value
        for text_id, value in ground_truth_raw.items()
    }

    return dataset, ground_truth


# ------------------------------------------------------------
# Benchmark corpora
# ------------------------------------------------------------

corpora = {
    name: load_corpus(name)
    for name in BENCHMARK_CORPORA
}


# ------------------------------------------------------------
# Unseen inference corpus
# ------------------------------------------------------------

inference_dataset, inference_ground_truth = load_corpus(
    INFERENCE_CORPUS
)


# ------------------------------------------------------------
# Sanity check
# ------------------------------------------------------------

for name, (dataset, ground_truth) in corpora.items():

    print(
        f"{name}: "
        f"{dataset.shape[0]:,} annotations | "
        f"{dataset['text_id'].nunique()} texts | "
        f"{len(ground_truth)} ground-truth entries"
    )


print(
    f"{INFERENCE_CORPUS}: "
    f"{inference_dataset.shape[0]:,} annotations | "
    f"{inference_dataset['text_id'].nunique()} texts | "
    f"{len(inference_ground_truth)} ground-truth entries"
)

A_default: 162,000 annotations | 100 texts | 100 ground-truth entries
B_weak_signal: 162,000 annotations | 100 texts | 100 ground-truth entries
C_deep: 162,000 annotations | 100 texts | 100 ground-truth entries
inference_unseen: 162,000 annotations | 100 texts


## 3. Hyperparameter search space

The benchmark explores the full 3,240-point configuration space defined by the Polarized Trees hyperparameters. We randomly sample 800 configurations and evaluate every sampled configuration on all three synthetic corpora, resulting in 2,400 benchmark runs.

The configuration with the highest mean Jaccard across A/B/C is selected for the subsequent experiments.

In [16]:
GRID = {
    "theta_filter": [0.2, 0.3, 0.4],

    "min_size_frac": [
        0.02,
        0.03,
        0.05,
    ],

    "max_depth": [
        4,
        6,
        8,
    ],

    "variant_beta": [
        ("max", 1.0),
        ("var", 1.0),
        ("beta", 0.5),
        ("beta", 1.0),
        ("beta", 2.0),
    ],

    "h": [
        0.05,
        0.10,
        0.15,
        0.20,
    ],

    "relative_h": [
        False,
        True,
    ],

    "theta_stop": [
        0.05,
        0.10,
        0.15,
    ],
}


grid_keys = list(GRID)

full_combinations = list(
    itertools.product(
        *(GRID[k] for k in grid_keys)
    )
)

print(
    "Full grid size:",
    len(full_combinations)
)


rng = random.Random(RANDOM_SEED)

sampled_combinations = rng.sample(
    full_combinations,
    min(
        N_SAMPLES,
        len(full_combinations),
    ),
)

print(
    "Sampled configurations:",
    len(sampled_combinations)
)

print(
    "Benchmark runs:",
    len(sampled_combinations)
    * len(BENCHMARK_CORPORA)
)

Full grid size: 3240
Sampled configurations: 800
Benchmark runs: 2400


## 4. Run the benchmark

Each sampled configuration is evaluated on the same three fixed synthetic corpora.

For every configuration–corpus pair, we record mean Jaccard, precision, recall, and exact match. These recovery metrics are available because the synthetic corpora provide the true active SCD dimensions for every text.

In [17]:
def run_one(config_tuple, corpus_name):

    params = dict(
        zip(
            grid_keys,
            config_tuple,
        )
    )

    variant, beta = params["variant_beta"]

    dataset, ground_truth = corpora[
        corpus_name
    ]

    pipe = PolarizedTreesPipeline(
        dims=DIMS,
        scale=SCALE,

        theta_filter=params[
            "theta_filter"
        ],

        min_size_frac=params[
            "min_size_frac"
        ],

        max_depth=params[
            "max_depth"
        ],

        variant=variant,
        beta=beta,

        h=params["h"],

        relative_h=params[
            "relative_h"
        ],

        theta_stop=params[
            "theta_stop"
        ],
    )

    out = pipe.run_full_evaluation(
        dataset,
        ground_truth=ground_truth,
        verbose=False,
    )

    recovery = out["recovery"]

    return {
        "corpus": corpus_name,

        "theta_filter":
            params["theta_filter"],

        "min_size_frac":
            params["min_size_frac"],

        "max_depth":
            params["max_depth"],

        "variant": variant,

        "beta": beta,

        "h": params["h"],

        "relative_h":
            params["relative_h"],

        "theta_stop":
            params["theta_stop"],

        "jaccard":
            recovery["jaccard"].mean(),

        "precision":
            recovery["precision"].mean(),

        "recall":
            recovery["recall"].mean(),

        "exact_match":
            recovery["exact_match"].mean(),
    }


jobs = [
    (
        combo,
        corpus_name,
    )
    for combo in sampled_combinations
    for corpus_name in BENCHMARK_CORPORA
]

print(
    "Jobs:",
    len(jobs)
)

Jobs: 2400


### Run the benchmark

The benchmark can be run serially or in parallel. Checkpointing is used so that intermediate results are saved during the experiment.

In [19]:
# tqdm is used only for benchmark progress reporting.
from tqdm.auto import tqdm

N_JOBS = max(
    1,
    (os.cpu_count() or 2) - 1,
)

CHECKPOINT_EVERY = 100

RESULTS_PATH = RESULTS_DIR / "benchmark_results.csv"

results = []
start = time.time()


def save_checkpoint(rows):
    pd.DataFrame(rows).to_csv(
        RESULTS_PATH,
        index=False,
    )


if N_JOBS == 1:

    for i, (
        combo,
        corpus_name,
    ) in enumerate(
        tqdm(
            jobs,
            total=len(jobs),
            desc="Benchmark",
        ),
        start=1,
    ):

        results.append(
            run_one(
                combo,
                corpus_name,
            )
        )

        if i % CHECKPOINT_EVERY == 0:
            save_checkpoint(results)

            elapsed = time.time() - start
            tqdm.write(
                f"Checkpoint: {i}/{len(jobs)} completed "
                f"({elapsed / 60:.1f} min)"
            )

else:

    with ProcessPoolExecutor(
        max_workers=N_JOBS
    ) as executor:

        futures = [
            executor.submit(
                run_one,
                combo,
                corpus_name,
            )
            for combo, corpus_name in jobs
        ]

        for i, future in enumerate(
            tqdm(
                as_completed(futures),
                total=len(futures),
                desc="Benchmark",
            ),
            start=1,
        ):

            results.append(
                future.result()
            )

            if i % CHECKPOINT_EVERY == 0:
                save_checkpoint(results)

                elapsed = time.time() - start
                tqdm.write(
                    f"Checkpoint: {i}/{len(jobs)} completed "
                    f"({elapsed / 60:.1f} min)"
                )


save_checkpoint(results)

elapsed = time.time() - start

print(
    f"Finished {len(results)} runs "
    f"in {elapsed / 60:.1f} minutes."
)


Benchmark:   0%|          | 0/2400 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 5. Model selection

We aggregate the results of each configuration across A/B/C and rank configurations by their mean Jaccard.

The highest-ranked configuration is selected for all subsequent experiments.

In [20]:
results_df = pd.DataFrame(
    results
)

CONFIG_COLS = [
    "theta_filter",
    "min_size_frac",
    "max_depth",
    "variant",
    "beta",
    "h",
    "relative_h",
    "theta_stop",
]


summary = (
    results_df
    .groupby(
        CONFIG_COLS,
        as_index=False,
    )
    [
        [
            "jaccard",
            "precision",
            "recall",
            "exact_match",
        ]
    ]
    .mean()
    .sort_values(
        "jaccard",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


print(
    "Top 20 configurations:"
)

display(
    summary.head(20)
)


best_config = (
    summary
    .iloc[0][CONFIG_COLS]
    .to_dict()
)


print(
    "\nSelected configuration:"
)

print(
    best_config
)

Top 20 configurations:


,theta_filter,min_size_frac,max_depth,variant,beta,h,relative_h,theta_stop,jaccard,precision,recall,exact_match
0,0.2,0.02,8,beta,1.0,0.15,True,0.10,0.888823,0.944651,0.932267,0.742754
1,0.3,0.02,6,var,1.0,0.10,True,0.10,0.879362,0.944913,0.930333,0.709646
2,0.3,0.02,6,beta,0.5,0.20,True,0.10,0.879188,0.938017,0.920742,0.724043
3,0.3,0.02,4,beta,2.0,0.15,True,0.10,0.876486,0.956571,0.907569,0.710829
4,0.4,0.02,8,beta,2.0,0.10,True,0.10,0.875095,0.942285,0.932810,0.682176
5,0.4,0.03,4,beta,2.0,0.10,True,0.10,0.875095,0.942285,0.932810,0.682176
6,0.2,0.03,4,beta,2.0,0.15,True,0.10,0.872662,0.952588,0.908169,0.705832
7,0.2,0.05,8,beta,2.0,0.10,True,0.10,0.869062,0.956464,0.907968,0.667874
8,0.4,0.02,4,beta,1.0,0.10,True,0.10,0.867799,0.928536,0.933516,0.679961
9,0.3,0.05,4,var,1.0,0.10,True,0.10,0.867034,0.956364,0.904613,0.657542



Selected configuration:
{'theta_filter': 0.2, 'min_size_frac': 0.02, 'max_depth': 8, 'variant': 'beta', 'beta': 1.0, 'h': 0.15, 'relative_h': True, 'theta_stop': 0.1}


## 6. Recovery performance of the selected configuration

We now evaluate the selected configuration separately on each synthetic corpus. The final row reports the mean across A/B/C, which is the quantity used for model selection.

In [21]:
best_rows = results_df.copy()

for key, value in best_config.items():

    best_rows = best_rows[
        best_rows[key] == value
    ]


selected_recovery = best_rows[
    [
        "corpus",
        "jaccard",
        "precision",
        "recall",
        "exact_match",
    ]
].copy()


mean_row = pd.DataFrame(
    [
        {
            "corpus": "Mean",

            "jaccard":
                selected_recovery[
                    "jaccard"
                ].mean(),

            "precision":
                selected_recovery[
                    "precision"
                ].mean(),

            "recall":
                selected_recovery[
                    "recall"
                ].mean(),

            "exact_match":
                selected_recovery[
                    "exact_match"
                ].mean(),
        }
    ]
)


selected_recovery = pd.concat(
    [
        selected_recovery,
        mean_row,
    ],
    ignore_index=True,
)


display(
    selected_recovery
)

,corpus,jaccard,precision,recall,exact_match
0,A_default,0.918651,0.974206,0.944444,0.797619
1,B_weak_signal,0.860317,0.866270,0.958333,0.702381
2,C_deep,0.887500,0.993478,0.894022,0.728261
3,Mean,0.888823,0.944651,0.932267,0.742754


## 7. Benchmark analyses

Beyond selecting the best configuration, we examine the behavior of the sampled configurations to understand which hyperparameter choices are associated with stronger recovery.

These analyses are descriptive and are not used to alter the selected configuration.

In [22]:
# Mean Jaccard by h and PRG normalization

prg_h = (
    results_df
    .groupby(
        [
            "relative_h",
            "h",
        ]
    )["jaccard"]
    .mean()
    .unstack(0)
)


display(
    prg_h.rename(
        columns={
            False:
                "Absolute PRG",
            True:
                "Normalized PRG",
        }
    )
)


# Top-20 configurations
top20 = summary.head(20)


print(
    "PRG variants among top 20:"
)

display(
    top20[
        "variant"
    ].value_counts()
)


# Frequency of individual
# hyperparameter values
for col in [
    "beta",
    "relative_h",
    "theta_stop",
    "h",
    "max_depth",
    "theta_filter",
    "min_size_frac",
]:

    print(
        f"\n{col}"
    )

    display(
        top20[col]
        .value_counts()
        .rename("count")
        .to_frame()
    )

relative_h,Absolute PRG,Normalized PRG
h,,
0.05,0.817412,0.821188
0.10,0.724609,0.836992
0.15,0.659908,0.824758
0.20,0.535699,0.821187


PRG variants among top 20:


,count
variant,
beta,16
var,4



beta


,count
beta,
1.0,8
2.0,7
0.5,5


...

,count
max_depth,
4,8
8,7
6,5



theta_filter


,count
theta_filter,
0.2,7
0.4,7
0.3,6



min_size_frac


,count
min_size_frac,
0.02,10
0.05,6
0.03,4


## 8. Inference on previously unseen synthetic data

The A/B/C corpora were used for model selection. We now apply the selected configuration to a separately generated synthetic corpus using a new random seed.

Ground truth is deliberately not provided to Polarized Trees. The resulting outputs are therefore the same ground-truth-free outputs that would be available on a real annotation dataset: Dimension Frequency (F), Subgroup Pole Consistency (C), Subgroup PRG (P), and inference diagnostics.

In [23]:
pipe = PolarizedTreesPipeline(
    dims=DIMS,
    scale=SCALE,

    theta_filter=
        best_config[
            "theta_filter"
        ],

    min_size_frac=
        best_config[
            "min_size_frac"
        ],

    max_depth=
        best_config[
            "max_depth"
        ],

    variant=
        best_config[
            "variant"
        ],

    beta=
        best_config[
            "beta"
        ],

    h=
        best_config[
            "h"
        ],

    relative_h=
        best_config[
            "relative_h"
        ],

    theta_stop=
        best_config[
            "theta_stop"
        ],
)


inference_results = (
    pipe.run_full_evaluation(
        inference_dataset,
        ground_truth=None,
        verbose=True,
    )
)


F = inference_results["F"]
C = inference_results["C"]
P = inference_results["P"]

diagnostics = (
    inference_results[
        "diagnostics"
    ]
)


print(
    "=== F: Dimension Frequency ==="
)

display(F)


print(
    "=== C: Subgroup Pole Consistency ==="
)

display(
    C.head(10)
)


print(
    "=== P: Subgroup PRG ==="
)

display(
    P.head(10)
)


print(
    "=== Diagnostics ==="
)

display(
    pd.Series(
        diagnostics,
        name="value",
    )
)

=== diagnostics ===
  retention_rate: 0.86
  mean_leaves: 7.593023255813954
  mean_depth: 2.3169984686064318
  mean_residual_ndfu: 0.17932784484016107
  mean_top_split_prg: 0.4408721547050139
  indeterminate_rate: 0.006125574272588055
  dims_never_used: []
=== F: Dimension Frequency ===


depth,1,2,3
dim,,,
age,16,22,21
education,17,29,20
gender,15,20,19
orientation,25,25,29
politics,13,25,27


=== C: Subgroup Pole Consistency ===


,n_s,frac_toxic,frac_civil
subgroup,,,
"((education, low),)",13,0.461538,0.538462
"((education, high),)",12,0.500000,0.500000
"((orientation, lgbtq+),)",12,0.500000,0.500000
"((orientation, heterosexual),)",12,0.583333,0.416667
"((education, medium),)",11,0.727273,0.272727
"((politics, center),)",8,0.500000,0.500000
"((politics, left),)",8,0.625000,0.375000
"((politics, right),)",7,0.714286,0.285714
"((age, 25-50), (education, high))",6,0.500000,0.500000


=== P: Subgroup PRG ===


,n_s,mean_prg
subgroup,,
"((age, >50), (gender, male), (orientation, lgbtq+))",2,0.679320
"((age, >50), (education, high), (politics, left))",1,0.643634
"((orientation, lgbtq+),)",12,0.635337
"((orientation, heterosexual),)",12,0.632869
"((education, low), (gender, non-binary), (politics, center))",1,0.623129
"((education, low), (gender, non-binary), (politics, right))",1,0.623129
"((education, low), (gender, female), (politics, right))",1,0.608743
"((education, low), (gender, female), (politics, center))",1,0.608743
"((education, low), (orientation, lgbtq+), (politics, center))",1,0.606818


=== Diagnostics ===


,value
retention_rate,0.86
mean_leaves,7.593023
mean_depth,2.316998
mean_residual_ndfu,0.179328
mean_top_split_prg,0.440872
indeterminate_rate,0.006126
dims_never_used,[]


## 9. Save the benchmark and inference outputs

The benchmark results, selected configuration, recovery results, and inference outputs are saved separately for reproducibility and downstream analysis.

In [24]:
# Save all benchmark and inference outputs inside RESULTS_DIR.

results_df.to_csv(
    RESULTS_DIR / "benchmark_runs.csv",
    index=False,
)

summary.to_csv(
    RESULTS_DIR / "benchmark_configuration_summary.csv",
    index=False,
)

selected_recovery.to_csv(
    RESULTS_DIR / "selected_configuration_recovery.csv",
    index=False,
)


with open(
    RESULTS_DIR / "selected_configuration.json",
    "w",
) as f:

    json.dump(
        best_config,
        f,
        indent=2,
        default=str,
    )


F.to_csv(
    RESULTS_DIR / "fcp_F_dimension_frequency.csv"
)

C.to_csv(
    RESULTS_DIR / "fcp_C_pole_consistency.csv"
)

P.to_csv(
    RESULTS_DIR / "fcp_P_subgroup_prg.csv"
)


with pd.ExcelWriter(
    RESULTS_DIR / "fcp_inference_results.xlsx"
) as writer:

    F.to_excel(
        writer,
        sheet_name="F_dimension_frequency",
    )

    C.to_excel(
        writer,
        sheet_name="C_pole_consistency",
    )

    P.to_excel(
        writer,
        sheet_name="P_subgroup_prg",
    )


print(
    "Saved benchmark and inference outputs to:",
    RESULTS_DIR.resolve(),
)


Saved benchmark and inference outputs.


In [26]:
# ------------------------------------------------------------
# Package benchmark outputs for transfer/archiving.
# ------------------------------------------------------------

import shutil
from pathlib import Path

if not RESULTS_DIR.exists():
    raise FileNotFoundError(
        f"{RESULTS_DIR} does not exist. "
        "Run the benchmark and save-output cells first."
    )

output_files = sorted(
    path
    for path in RESULTS_DIR.iterdir()
    if path.is_file()
)

if not output_files:
    raise FileNotFoundError(
        f"No benchmark outputs found in {RESULTS_DIR}."
    )

print("Files included in benchmark_results.zip:")
for path in output_files:
    print(f"  {path.name}")

ZIP_PATH = shutil.make_archive(
    str(PROJECT_ROOT / "benchmark_results"),
    "zip",
    root_dir=PROJECT_ROOT,
    base_dir=RESULTS_DIR.name,
)

print(f"\nCreated: {ZIP_PATH}")
print(
    f"Size: {Path(ZIP_PATH).stat().st_size / 1024**2:.2f} MB"
)
print("The ZIP is in the project root.")


Files included in download:
  A_default_config.json
  A_default_dataset.csv
  A_default_ground_truth.json
  B_weak_signal_config.json
  B_weak_signal_dataset.csv
  B_weak_signal_ground_truth.json
  C_deep_config.json
  C_deep_dataset.csv
  C_deep_ground_truth.json
  benchmark_configuration_summary.csv
  benchmark_results.csv
  benchmark_runs.csv
  fcp_C_pole_consistency.csv
  fcp_F_dimension_frequency.csv
  fcp_P_subgroup_prg.csv
  fcp_inference_results.xlsx
  inference_unseen_config.json
  inference_unseen_dataset.csv
  inference_unseen_ground_truth.json
  selected_configuration.json
  selected_configuration_recovery.csv


RuntimeError: File size too large, try using force_zip64